# CSV vs Redis DB2 ("past_signal") — strategy_lab

Đối chiếu dữ liệu quá khứ (full-history) mà `strategy_lab`'s continuous worker (`og_signal/redis_io/worker.py`) đẩy lên Redis DB2 (`redis.past_signal`, key_prefix `L_PastSignal`) với chính CSV export (`runtime/exports/`) của cùng hệ thống — 2 đường xuất dữ liệu của cùng 1 nguồn tính toán, phải khớp tuyệt đối vì HASH dùng đúng schema cột CSV (`export_cli.signal_frame()`), không cần quy đổi gì.

Chỉ đọc — không ghi gì lên Redis/đĩa.

**Lưu ý quy mô**: DB2 hiện có nhiều pair hơn hẳn CSV (bao gồm cả các symbol forex ngoài phạm vi `runtime/exports/` hiện tại — DB2 phục vụ scope backtest rộng hơn của `strategy_lab`). Notebook này đi từ phía CSV (88 file, 11 symbol) — với mỗi pair CSV đang có, kiểm DB2 đã có dữ liệu chưa, nếu có thì so khớp toàn bộ lịch sử (không chỉ đoạn gần đây, vì DB2 không TTL/LTRIM, luôn giữ full history).

In [1]:
from __future__ import annotations

import sys
from pathlib import Path

import pandas as pd

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "research" else Path.cwd()
sys.path.insert(0, str(REPO_ROOT))

from strategy_lab.src.og_signal.redis_io.client import create_client, load_config

EXPORTS_DIR = REPO_ROOT / "runtime" / "exports"

config = load_config()
past_signal_config = config["redis"]["past_signal"]
db2 = create_client(config["redis"], past_signal_config["db"])
db2.ping()
print(f"Đã kết nối Redis DB{past_signal_config['db']} (past_signal) — key_prefix={past_signal_config['key_prefix']!r}")

Đã kết nối Redis DB2 (past_signal) — key_prefix='L_PastSignal'


In [2]:
# Liệt kê các pair (strategy, symbol, timeframe) đang có CSV trong runtime/exports/.
# Chỉ tính file mode "nt" (no_trend) -- khớp trend_filter_enabled=false hiện tại.
# strategy tự có dấu "_" ("ma_cross") nên khớp theo tiền tố đã biết trước,
# không split "_" ngây thơ (cùng lý do CLAUDE.md mục 7).
_STRATEGY_PREFIXES = ["ma_cross_", "combo_"]  # dài hơn/đặc hiệu hơn trước


def parse_csv_filename(name: str) -> tuple[str, str, str] | None:
    suffix = "_full_history_nt.csv"
    if not name.endswith(suffix):
        return None
    body = name[: -len(suffix)]
    for prefix in _STRATEGY_PREFIXES:
        if body.startswith(prefix):
            strategy = prefix.rstrip("_")
            symbol_tf = body[len(prefix):]
            if "_" not in symbol_tf:
                return None
            symbol, timeframe = symbol_tf.rsplit("_", 1)
            return strategy, symbol, timeframe
    return None


csv_pairs = []
for path in sorted(EXPORTS_DIR.glob("*_full_history_nt.csv")):
    parsed = parse_csv_filename(path.name)
    if parsed is not None:
        csv_pairs.append(parsed)

print(f"CSV: {len(csv_pairs)} pair trong runtime/exports/ (mode no_trend)")

CSV: 88 pair trong runtime/exports/ (mode no_trend)


In [3]:
# Với mỗi pair CSV đang có, kiểm DB2 đã có dữ liệu chưa; nếu có thì đọc
# toàn bộ lịch sử (LRANGE full list + pipeline HGETALL từng dòng).
def db2_list_key(key_prefix: str, symbol: str, timeframe: str, strategy: str) -> str:
    return f"{key_prefix}_{symbol.upper()}_{timeframe.upper()}_{strategy.upper()}"


db2_frames = []
pairs_missing_on_db2 = []

for strategy, symbol, timeframe in csv_pairs:
    list_key = db2_list_key(past_signal_config["key_prefix"], symbol, timeframe, strategy)
    stamps = db2.lrange(list_key, 0, -1)
    if not stamps:
        pairs_missing_on_db2.append((strategy, symbol, timeframe))
        continue
    pipe = db2.pipeline(transaction=False)
    for stamp in stamps:
        pipe.hgetall(f"{list_key}:{stamp}")
    hashes = pipe.execute()
    rows = [h for h in hashes if h]
    if not rows:
        pairs_missing_on_db2.append((strategy, symbol, timeframe))
        continue
    frame = pd.DataFrame(rows).assign(strategy=strategy, symbol=symbol, timeframe=timeframe)
    db2_frames.append(frame)

df_db2 = pd.concat(db2_frames, ignore_index=True) if db2_frames else pd.DataFrame()
if not df_db2.empty:
    df_db2["signal"] = pd.to_numeric(df_db2["signal"], errors="coerce").astype("Int64")
    df_db2["atr"] = pd.to_numeric(df_db2["atr"], errors="coerce")
    if "entry" in df_db2.columns:
        df_db2["entry"] = pd.to_numeric(df_db2["entry"], errors="coerce")

n_pairs_on_db2 = len(csv_pairs) - len(pairs_missing_on_db2)
print(f"DB2: {n_pairs_on_db2}/{len(csv_pairs)} pair CSV đã có dữ liệu trên DB2, tổng {len(df_db2)} dòng")
if pairs_missing_on_db2:
    print(f"Chưa có trên DB2 ({len(pairs_missing_on_db2)} pair, worker có thể chưa chạy tới):")
    for p in pairs_missing_on_db2:
        print("  -", p)

DB2: 88/88 pair CSV đã có dữ liệu trên DB2, tổng 71435 dòng


In [4]:
# Đọc CSV cho đúng các pair đã có trên DB2 (bỏ qua pair chưa có, không so được).
def read_export_csv(strategy: str, symbol: str, timeframe: str) -> pd.DataFrame:
    path = EXPORTS_DIR / f"{strategy}_{symbol}_{timeframe}_full_history_nt.csv"
    df = pd.read_csv(path)
    if "entry" not in df.columns:
        df["entry"] = None
    return df


csv_frames = []
if not df_db2.empty:
    for (strategy, symbol, timeframe), _ in df_db2.groupby(["strategy", "symbol", "timeframe"]):
        csv_df = read_export_csv(strategy, symbol, timeframe)
        csv_frames.append(csv_df.assign(strategy=strategy, symbol=symbol, timeframe=timeframe))

df_csv = pd.concat(csv_frames, ignore_index=True) if csv_frames else pd.DataFrame()
if not df_csv.empty:
    df_csv["entry"] = pd.to_numeric(df_csv["entry"], errors="coerce") if "entry" in df_csv.columns else None

print(f"CSV: {len(df_csv)} dòng đọc lại cho {n_pairs_on_db2} pair đang có trên DB2")

CSV: 225392 dòng đọc lại cho 88 pair đang có trên DB2


In [5]:
# Đối chiếu theo (strategy, symbol, timeframe, bartime) -- outer join để bắt
# được cả dòng chỉ có 1 bên (thiếu bên kia) lẫn dòng có ở cả 2 nhưng giá trị lệch.
if df_db2.empty:
    print("Chưa có pair nào để đối chiếu.")
    comparison = pd.DataFrame()
else:
    comparison = df_db2.merge(
        df_csv,
        on=["strategy", "symbol", "timeframe", "bartime"],
        how="outer",
        suffixes=("_db2", "_csv"),
        indicator=True,
    )
    comparison["signal_match"] = comparison["signal_db2"] == comparison["signal_csv"]
    comparison["entry_match"] = comparison.apply(
        lambda r: True if pd.isna(r.get("entry_db2")) or pd.isna(r.get("entry_csv"))
        else abs(r["entry_db2"] - r["entry_csv"]) < 0.01,
        axis=1,
    )
    comparison["atr_match"] = (comparison["atr_db2"] - comparison["atr_csv"]).abs() < 0.01

comparison.head() if not comparison.empty else comparison

,bartime,atr_db2,entry_db2,signal_db2,strategy,symbol,timeframe,atr_csv,entry_csv,signal_csv,_merge,signal_match,entry_match,atr_match
0,2017-05-02 18:00,NaN,NaN,<NA>,combo,BTCUSD,H1,22.893009,1482.44,2.0,right_only,<NA>,True,False
1,2017-05-03 05:00,NaN,NaN,<NA>,combo,BTCUSD,H1,16.904690,1506.00,2.0,right_only,<NA>,True,False
2,2017-05-03 09:00,NaN,NaN,<NA>,combo,BTCUSD,H1,15.817761,1625.74,1.0,right_only,<NA>,True,False
3,2017-05-03 13:00,NaN,NaN,<NA>,combo,BTCUSD,H1,15.602283,1500.00,2.0,right_only,<NA>,True,False
4,2017-05-03 15:00,NaN,NaN,<NA>,combo,BTCUSD,H1,17.083061,1625.00,1.0,right_only,<NA>,True,False


In [6]:
# Tổng kết.
if comparison.empty:
    print("Không có gì để tổng kết.")
else:
    both = comparison[comparison["_merge"] == "both"]
    only_db2 = comparison[comparison["_merge"] == "left_only"]
    only_csv = comparison[comparison["_merge"] == "right_only"]

    print(f"Tổng {len(comparison)} bartime duy nhất trên {n_pairs_on_db2} pair đã đối chiếu.")
    print(f"  - Có ở CẢ HAI bên: {len(both)}")
    print(f"  - Chỉ có trên DB2 (CSV chưa có dòng này): {len(only_db2)}")
    print(f"  - Chỉ có trên CSV (DB2 chưa có dòng này): {len(only_csv)}")

    if len(both):
        side_ok = int(both["signal_match"].sum())
        entry_ok = int(both["entry_match"].sum())
        atr_ok = int(both["atr_match"].sum())
        print(f"\nTrong {len(both)} dòng có ở cả 2 bên:")
        print(f"  - Khớp signal: {side_ok}/{len(both)}")
        print(f"  - Khớp entry (chỉ combo có cả 2 bên): {entry_ok}/{len(both)}")
        print(f"  - Khớp atr: {atr_ok}/{len(both)}")

        mismatches = both[~both["signal_match"] | ~both["entry_match"] | ~both["atr_match"]]
        if not mismatches.empty:
            print("\nCÁC DÒNG LỆCH GIÁ TRỊ:")
            display(mismatches[[
                "strategy", "symbol", "timeframe", "bartime",
                "signal_db2", "signal_csv", "entry_db2", "entry_csv", "atr_db2", "atr_csv",
            ]])
        else:
            print("\nKhông có dòng nào lệch giá trị.")

    if not only_db2.empty:
        print(f"\n{len(only_db2)} dòng chỉ có trên DB2, không có trong CSV (theo pair):")
        display(only_db2.groupby(["strategy", "symbol", "timeframe"]).size().rename("so_dong"))
    if not only_csv.empty:
        print(f"\n{len(only_csv)} dòng chỉ có trong CSV, chưa có trên DB2 (theo pair):")
        display(only_csv.groupby(["strategy", "symbol", "timeframe"]).size().rename("so_dong"))

Tổng 225605 bartime duy nhất trên 88 pair đã đối chiếu.
  - Có ở CẢ HAI bên: 71222
  - Chỉ có trên DB2 (CSV chưa có dòng này): 213
  - Chỉ có trên CSV (DB2 chưa có dòng này): 154170

Trong 71222 dòng có ở cả 2 bên:
  - Khớp signal: 71222/71222
  - Khớp entry (chỉ combo có cả 2 bên): 71222/71222
  - Khớp atr: 71222/71222

Không có dòng nào lệch giá trị.

213 dòng chỉ có trên DB2, không có trong CSV (theo pair):


strategy  symbol  timeframe
combo     BTCUSD  H1           2
          DE40    H1           3
                  H4           1
          FR40    H1           1
                  H3           1
                              ..
ma_cross  US30    M45          2
          US500   M10          8
                  M20          4
                  M30          5
                  M45          2
Name: so_dong, Length: 65, dtype: int64


154170 dòng chỉ có trong CSV, chưa có trên DB2 (theo pair):


strategy  symbol  timeframe
combo     BTCUSD  H1           4203
                  H2           1162
                  H3           1187
                  H4           1161
          DE40    H1           2626
                               ... 
ma_cross  US30    M45          1345
          US500   M10          2980
                  M20          3096
                  M30          2074
                  M45          1349
Name: so_dong, Length: 88, dtype: int64